# Biodiversidad Forestal y Gestion Ambiental en Michoacan
## HackODS UNAM 2026 ODS 15: Vida de Ecosistemas Terrestres



En este cuaderno se analiza la composicion del aprovechamiento maderable y la gestion ambiental en Michoacan (Entidad 16), como indicadores de la presion humana sobre los ecosistemas terrestres.

**Fuentes de datos:**
- biodiversidad_especies_ods15.csv — Volumen de produccion forestal maderable por especie (Pino, Encino, Otras)
- gestion_ambiental_ods15.csv — Unidades de produccion con acciones ambientales (con y sin apoyo gubernamental)

In [62]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Estilo visual del proyecto
template_style = 'plotly_white'
COLOR_PINO   = '#2E8B57'
COLOR_ENCINO = '#8B4513'
COLOR_OTRAS  = '#DAA520'
COLOR_APOYO  = '#4CAF50'
COLOR_SIN    = '#F44336'
COLOR_MICH   = '#FF6F00'

print('Librerias cargadas correctamente.')

Librerias cargadas correctamente.


In [63]:
# Carga de datos
df_bio     = pd.read_csv('../data/procesados/biodiversidad_especies_ods15.csv')
df_gestion = pd.read_csv('../data/procesados/gestion_ambiental_ods15.csv')

# Limpieza: extraer nombre corto de la entidad
df_bio['estado'] = df_bio['entidad'].str.replace(r'^[0-9]+\s+', '', regex=True).str.strip()

# Columna de total maderable
df_bio['Total_Maderable'] = df_bio['Pino'] + df_bio['Encino'] + df_bio['Otras']

# Filtros Michoacan
mich_bio     = df_bio[df_bio['estado'] == 'MIC']
mich_gestion = df_gestion[df_gestion['entidad'].str.contains('Michoacán', na=False)]

print(f'Entidades en biodiversidad: {len(df_bio)}')
print(f'Entidades en gestion ambiental: {len(df_gestion)}')
print(f'Michoacan - Total maderable: {mich_bio["Total_Maderable"].values[0]:,.2f} m3')

Entidades en biodiversidad: 31
Entidades en gestion ambiental: 32
Michoacan - Total maderable: 328,442.66 m3


---
## 1. Composicion del Aprovechamiento Maderable en Michoacan

El grafico de dona muestra la distribucion del volumen extraido por tipo de especie. La dominancia del Pino es significativa ya que es la misma especie que ocupa los rangos altitudinales donde el aguacate se expande, lo cual evidencia un conflicto directo de uso de suelo.

In [64]:
# Grafico 1: Dona - Composicion maderable en Michoacan
especies = ['Pino', 'Encino', 'Otras']
valores_mich = mich_bio[especies].iloc[0].values

fig1 = px.pie(
    names=especies,
    values=valores_mich,
    title='Proporcion del Volumen Maderable Extraido por Especie<br><sup>Michoacan - Datos procesados ODS 15</sup>',
    color_discrete_sequence=[COLOR_PINO, COLOR_ENCINO, COLOR_OTRAS],
    hole=0.45
)

fig1.update_traces(
    textposition='inside',
    textinfo='percent+label',
    marker=dict(line=dict(color='#FFFFFF', width=2))
)

fig1.update_layout(
    template=template_style,
    height=500,
    annotations=[dict(
        text='Especies<br>Michoacan',
        x=0.5, y=0.5,
        font_size=14,
        showarrow=False
    )]
)

fig1.show()

---
## 2. Comparativa Nacional: Top 10 Entidades por Aprovechamiento Forestal

Ademas de la exportacion de aguacate, Michoacan tambien extrae volumenes significativos de madera. A continuacion se muestra su posicion en el ranking nacional.

In [65]:
# Grafico 2: Barras apiladas - Top 10 entidades
df_bio_sin_nal = df_bio[df_bio['estado'] != 'NAL'].copy()
df_bio_top = df_bio_sin_nal.sort_values('Total_Maderable', ascending=False).head(10)

fig2 = px.bar(
    df_bio_top,
    x='estado',
    y=['Pino', 'Encino', 'Otras'],
    title='Top 10 Entidades por Volumen de Aprovechamiento Maderable<br><sup>Distribuido por especie - Fuente: biodiversidad_especies_ods15</sup>',
    labels={'value': 'Volumen (m3 en rollo)', 'estado': 'Entidad', 'variable': 'Especie'},
    color_discrete_map={'Pino': COLOR_PINO, 'Encino': COLOR_ENCINO, 'Otras': COLOR_OTRAS}
)

fig2.update_layout(
    template=template_style,
    barmode='stack',
    xaxis_tickangle=-45,
    height=550,
    legend_title_text='Especie',
    yaxis_title='Volumen Total (m3)'
)

if 'MIC' in df_bio_top['estado'].values:
    y_val = df_bio_top[df_bio_top['estado'] == 'MIC']['Total_Maderable'].values[0]
    fig2.add_annotation(
        x='MIC', y=y_val + (y_val * 0.08),
        text='Michoacan',
        showarrow=True, arrowhead=2, ax=0, ay=-40,
        font=dict(color=COLOR_MICH, size=13)
    )

fig2.show()

Durango y Chihuahua dominan la produccion forestal a nivel nacional, pero Michoacan aparece como un extractor relevante (especialmente de Pino), lo cual intensifica la presion sobre sus ecosistemas ya amenazados por la expansion agricola.

---
## 3. Gestion Ambiental: Acciones con y sin apoyo gubernamental

Se comparan las unidades de produccion que realizan acciones de gestion ambiental.

In [66]:
# Grafico 3: Barras agrupadas - Gestion ambiental Top 15
df_gestion_top = df_gestion.sort_values('unidades_con_acciones', ascending=False).head(15)

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    x=df_gestion_top['entidad'],
    y=df_gestion_top['unidades_con_apoyo'],
    name='Con Apoyo Gubernamental',
    marker_color=COLOR_APOYO
))

fig3.add_trace(go.Bar(
    x=df_gestion_top['entidad'],
    y=df_gestion_top['unidades_sin_apoyo'],
    name='Sin Apoyo (esfuerzo propio)',
    marker_color=COLOR_SIN
))

fig3.update_layout(
    title='Unidades de Produccion con Acciones de Gestion Ambiental<br><sup>Top 15 Entidades - Con apoyo vs. Sin apoyo gubernamental</sup>',
    xaxis_tickangle=-45,
    barmode='stack',
    template=template_style,
    legend_title='Tipo de Financiamiento',
    yaxis_title='Numero de Unidades de Produccion',
    height=600
)

if 'Michoacán de Ocampo' in df_gestion_top['entidad'].values:
    row = df_gestion_top[df_gestion_top['entidad'].str.contains('Michoacán')].iloc[0]
    fig3.add_annotation(
        x=row['entidad'],
        y=row['unidades_con_acciones'] + 20,
        text='80.7% sin apoyo',
        showarrow=True, arrowhead=2, ax=0, ay=-35,
        font=dict(color=COLOR_SIN, size=12)
    )

fig3.show()

En Michoacan, mas del 80% de las unidades que realizan acciones ambientales lo hacen sin apoyo gubernamental. Esto revela una situacion contradictoria: la entidad con mayor presion por deforestacion agropecuaria es una de las menos apoyadas institucionalmente.

---
## 4. Foco en Michoacan: Radiografia de la Gestion Ambiental

Se presenta un analisis detallado del esfuerzo comunitario contra el apoyo institucional, con indicadores clave.

In [67]:
# Grafico 4: Dona detallada y KPIs de Michoacan
mich_gest = mich_gestion.iloc[0]

fig4 = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
    subplot_titles=(
        'Financiamiento de Acciones Ambientales',
        'Indicadores Clave'
    )
)

# Panel izquierdo: Dona
fig4.add_trace(
    go.Pie(
        labels=['Con Apoyo', 'Sin Apoyo'],
        values=[mich_gest['unidades_con_apoyo'], mich_gest['unidades_sin_apoyo']],
        marker=dict(
            colors=[COLOR_APOYO, COLOR_SIN],
            line=dict(color='#FFFFFF', width=2)
        ),
        hole=0.4,
        textinfo='percent+label+value',
        textposition='inside'
    ),
    row=1, col=1
)

# Panel derecho: KPIs en barras horizontales
kpi_names = ['Total<br>Unidades', 'Con<br>Acciones', 'Con<br>Apoyo', 'Sin<br>Apoyo']
kpi_vals  = [
    mich_gest['total_unidades'],
    mich_gest['unidades_con_acciones'],
    mich_gest['unidades_con_apoyo'],
    mich_gest['unidades_sin_apoyo']
]
kpi_colors = ['#78909C', '#42A5F5', COLOR_APOYO, COLOR_SIN]

fig4.add_trace(
    go.Bar(
        y=kpi_names, x=kpi_vals,
        orientation='h',
        marker_color=kpi_colors,
        text=kpi_vals, textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

fig4.update_layout(
    template=template_style,
    height=450,
    title_text='Michoacan de Ocampo - Gestion Ambiental en Unidades de Produccion',
    showlegend=False
)

fig4.show()

